In [46]:
import sqlite3
import pandas as pd

# built into Python
# for readable output

con = sqlite3.connect(r'..\\data\\raw\\airtel.db') # two dots: notebooks/

def q(sql):
    return pd.read_sql(sql, con)

# the same helper as
# the IPL project

In [47]:
def run_sql_file(path):                         # one .sql file
    with open(path, 'r') as f:                  # read only
        script = f.read()                       # whole file
    con.executescript(script)                   # every statement
    con.commit()                                # save to the .db
    print('ran', path)                          # show what ran
    
run_sql_file(r'..\sql\00_test.sql')              # apply the file
q('SELECT COUNT(*) AS n FROM customers;')       # check it worke

ran ..\sql\00_test.sql


,n
0,22000


In [48]:
run_sql_file(r'..\sql\01_revenue_customer.sql')

ran ..\sql\01_revenue_customer.sql


In [49]:
q('''
    SELECT
    COUNT(*) AS rows_,
    SUM(recharge_count = 0) AS paid_zero,
    ROUND(AVG(actual_arpu_6m), 2) AS actual,
    (
        SELECT ROUND(AVG(arpu_inr), 2)
        FROM customers
    ) AS stated
FROM v_revenue_customer;
    ''')

,rows_,paid_zero,actual,stated
0,22000,3711,344.46,421.65


In [50]:
q('''SELECT *
FROM v_revenue_customer
WHERE customer_id IN (4, 9);
''')

,customer_id,recharge_count,recharge_total_6m,actual_arpu_6m
0,4,0,0,0.00
1,9,5,1745,290.83


In [51]:
run_sql_file(r'..\sql\02_usage_customer.sql')

ran ..\sql\02_usage_customer.sql


In [52]:
q('''
SELECT
    COUNT(*) AS total_customers,

    SUM(usage_days = 0) AS no_usage_customers,

    ROUND(AVG(mb_per_active_day), 2)
        AS avg_mb_per_active_day,

    ROUND(AVG(mb_per_observed_day), 2)
        AS avg_mb_per_observed_day

FROM v_usage_customer;
''')

,total_customers,no_usage_customers,avg_mb_per_active_day,avg_mb_per_observed_day
0,22000,4,1404.44,524.05


In [53]:
q('''
SELECT *
FROM v_usage_customer
WHERE customer_id IN (9, 266);
''')

,customer_id,usage_days,total_data_mb,total_voice_min,total_sms,roaming_days,mb_per_active_day,mb_per_observed_day,last_usage_date,recency_days
0,9,65,100065.3,3292,43,3,1539.47,552.85,2025-06-28,2.0
1,266,0,0.0,0,0,0,NaN,0.00,NaN,NaN


In [54]:
run_sql_file(r'..\sql\03_complaint_customer.sql')

ran ..\sql\03_complaint_customer.sql


In [55]:
q('''
SELECT
    COUNT(*) AS total_customers,

    SUM(complaint_count = 0) AS never_complained,

    SUM(
        complaint_count > 0
        AND unresolved_count = 0
    ) AS all_resolved,

    SUM(unresolved_count > 0) AS has_open_complaints

FROM v_complaint_customer;
''')

,total_customers,never_complained,all_resolved,has_open_complaints
0,22000,6631,8689,6680


In [56]:
q('''
SELECT *
FROM v_complaint_customer
WHERE customer_id IN (9, 15);
''')

,customer_id,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,9,3,0,3,9.33
1,15,1,1,0,NaN


In [57]:
run_sql_file(r'..\sql\04_plans_clean.sql')

ran ..\sql\04_plans_clean.sql


In [58]:
q('''
SELECT *
FROM v_plans_clean;
''')

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,quota_basis,quota_gb,monthly_gb,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,per day,0.5,15.0,NaN,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,per day,1.0,30.0,NaN,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,per day,1.0,30.0,NaN,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,per day,1.5,45.0,NaN,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,per day,2.5,75.0,NaN,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,per day,2.5,75.0,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,per day,1.5,45.0,NaN,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,per day,2.0,60.0,NaN,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,per day,2.0,60.0,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,per month,40.0,40.0,OTT-Lite,1


In [59]:
q('''
SELECT
    plan_id,
    plan_type,
    data_quota,
    quota_basis,
    monthly_gb
FROM v_plans_clean
ORDER BY plan_id;
''')

,plan_id,plan_type,data_quota,quota_basis,monthly_gb
0,1,Prepaid,0.5 GB/day,per day,15.0
1,2,Prepaid,1 GB/day,per day,30.0
2,3,Prepaid,1 GB/day,per day,30.0
3,4,Prepaid,1.5 GB/day,per day,45.0
4,5,Prepaid,2.5 GB/day,per day,75.0
5,6,Prepaid,2.5 GB/day,per day,75.0
6,7,Prepaid,1.5 GB/day,per day,45.0
7,8,Prepaid,2 GB/day,per day,60.0
8,9,Prepaid,2 GB/day,per day,60.0
9,10,Postpaid,40 GB/month,per month,40.0


In [60]:
import sqlite3 
conn = sqlite3.connect(r"C:\Users\navee\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db") 
with open(r"C:\Users\navee\OneDrive\Desktop\airtel-telecom-analytics\sql\05_usage_month.sql", "r") as file:    
    sql_script = file.read() 
    conn.executescript(sql_script) 
    conn.commit() 
    print("v_usage_month created successfully!") 
    conn.close()

v_usage_month created successfully!


In [61]:
q('''
    SELECT COUNT(*) AS total_rows
    FROM v_usage_month;
''')

,total_rows
0,117832


In [62]:
import sqlite3 
conn = sqlite3.connect(r"..\data\raw\airtel.db") 
with open(r"..\sql\06_customer_features.sql", "r") as file:    
    sql_script = file.read() 
    conn.executescript(sql_script) 
    conn.commit() 
    print("customer_features created successfully!") 
    conn.close()

customer_features created successfully!


In [63]:
q('''
SELECT COUNT(*) AS total_subscribers
FROM customer_features;
''')

,total_subscribers
0,22000


In [64]:
q('''
SELECT *
FROM customer_features
WHERE customer_id = 9;
''')

,customer_id,activation_date,tenure_days,tenure_years,observed_days,circle_name,region,plan_name,plan_type,price_inr,...,recency_days,recharge_count,recharge_total_6m,actual_arpu_6m,stated_arpu,arpu_gap,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,9,2022-09-04,1030,2,180,Delhi,North,Prepaid Data 349,Prepaid,349,...,2,5,1745,290.83,374,83.17,3,0,3,9.33
